# 04_serving: Publish Gold to Azure SQL for Power BI

**Input:** 5 Gold Delta tables in `gold/`
**Output:** the same tables in Azure SQL, schema `gold` (overwrite on every run → idempotent)
**Credentials:** SQL password read from Azure Key Vault via the Databricks secret scope `kv-ecommerce`, never stored in code

Why a serving layer: Power BI connects natively to SQL, and the database is free and always reachable without a Spark cluster running.

In [0]:
from pyspark.sql import functions as F

GOLD_PATH = "abfss://gold@ecommercedatalakebp01.dfs.core.windows.net"

# loginTimeout=60 gives the serverless database time to wake up from auto-pause
JDBC_URL = (
    "jdbc:sqlserver://ecommerce-sql-server-bp01.database.windows.net:1433;"
    "database=ecommerce-db;"
    "encrypt=true;trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=60;"
)
SQL_USER = "sqladmin"
SQL_PASSWORD = dbutils.secrets.get(scope="kv-ecommerce", key="sql-password")  # shows as [REDACTED] if printed

In [0]:
# Metadata-driven: add a table name here to publish it; dimensions first, then facts
GOLD_TABLES = ["dim_customers", "dim_products", "dim_date", "fact_orders", "fact_events"]

for table in GOLD_TABLES:
    df = spark.read.format("delta").load(f"{GOLD_PATH}/{table}")

    (
        df.write
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", f"gold.{table}")
        .option("user", SQL_USER)
        .option("password", SQL_PASSWORD)
        .mode("overwrite")
        .save()
    )
    print(f"{table}: {df.count()} rows written")